In [ ]:
#https://www.epochconverter.com/
#numpy 1.22.1, transformers 4.35, cuda 11.8

from telethon import TelegramClient
import asyncio
from datetime import timezone,datetime,timedelta
import os
import pandas as pd
import pickle
import time
from cleantext import clean
import re
import calendar
from os import listdir
from os.path import isfile, join
from ast import literal_eval

import spacy
import nltk
from nltk.stem.porter import *      #rip spacy did not have Porter Stemmer
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from scipy.stats import mannwhitneyu
from scipy.stats import kruskal

import torch
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

from Crypto.Cipher import AES

api_id=9999999  #example
api_hash="79873985fhiu239789"     #example api_hash

#Delete lines below when you don't need decryption for api_id,api_hash
lockedapi_id=b"\xeb\xf0*\x17\x89\xe78\xfe"
lockedapi_hash=b"\xa4\xf62J\x02 \xcc\xd2\x14\x88\xb5\xe5\xc1\xc6/\xae\xcd\xaf\x908\xc5G\x8ea.\xcca4S\x86\x81\xbf"
try:
    with open(".\\key.bin",mode="rb") as output:
        key=output.readline().rstrip(b"\n")
        idnonce=output.readline().rstrip(b"\n")
        hashnonce=output.readline().rstrip(b"\n")
except:
    raise FileNotFoundError("please change the api_id,api_hash and delete these lines, or find the correct key")

idcipher=AES.new(key,AES.MODE_EAX,nonce=idnonce)
api_id=int(idcipher.decrypt(lockedapi_id).decode())

hashcipher=AES.new(key, AES.MODE_EAX,nonce=hashnonce)
api_hash=hashcipher.decrypt(lockedapi_hash).decode()
#delete ends here

In [ ]:
#Check if cuda exists
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.current_device())
print(torch.cuda.get_device_name(0))

In [ ]:
async def scrap(apiid,apihash,group,timestart,timeend,who="",searchmsg="",output_file="rawdata.ob"):
    outputlist=[]
    #do the date change format here   
    timestart=datetime.fromisoformat(timestart).replace(tzinfo=timezone.utc)    #UTC time
    timeend=datetime.fromisoformat(timeend).replace(tzinfo=timezone.utc)        #UTC time

    async with TelegramClient('', apiid, apihash) as client:
        async for message in client.iter_messages(group,search=searchmsg,from_user=who):
            if timestart <= message.date <= timeend:
                cleantext=clean(message.text, no_emoji=True)
                outputlist.append({
                    "time":message.date,
                    "content":cleantext,
                    "sender_id":message.from_id
                })
            elif timestart>message.date:
                break

    #Output saved list
    with open(f"data\\{output_file}","wb") as output:
        pickle.dump(outputlist, output)

async def scrapwithnofromuser(apiid,apihash,group,timestart,timeend,searchmsg="",output_file="test.ob"):         #For scrapping user's chat, have some changes
    outputlist=[]
    #do the date change format here
    timestart=datetime.fromisoformat(timestart).replace(tzinfo=timezone.utc)    #UTC time
    timeend=datetime.fromisoformat(timeend).replace(tzinfo=timezone.utc)        #UTC time

    async with TelegramClient('collectchat', apiid, apihash) as client:
        async for message in client.iter_messages(group,search=searchmsg,offset_date=timeend):
            if timestart <= message.date <= timeend:
                cleantext=clean(message.text, no_emoji=False)       #False, as want to keep the emojis for investigation (emoji also useful for role claim)
                outputlist.append({
                    "time":message.date,
                    "content":cleantext,
                    "sender_id":message.from_id
                })
            elif timestart>message.date:
                break

    #Output saved list
    with open(f"data\\{output_file}","wb") as output:
        pickle.dump(outputlist, output)

In [ ]:
#First goal: get game counts within a period, get win/loss for each of the player!
#Don't need to run this if you have collected things needed already
group="@WerewolfAmnesia"
datemin="2025-01-01"        #At most 2024-02-01, for thief changing
datemax="2025-03-08"
searchmessage="Game Length:"
findperson="@werewolfbetabot"

if not (os.path.isdir("data")):
    os.mkdir("data")     #making corresponding dir

await scrap(api_id,api_hash,group,datemin,datemax,findperson,searchmessage)     #get the results for the determined period, save as rawdata.ob (rip should have a better name)

Working starts from here:

In [ ]:
#Read the file saved, extract things you need
#extract the time period for each game, so that you can extract msg during this given time period.
#About the game id, you can hash it with the group name if you want to make it capable with more than 1 groups
def get_all_integers(s):
    return [int(num) for num in re.findall(r'\d+', s)]

def get_time(string):
    pattern=r'\d{2}:\d{2}:\d{2}'  # Matches HH:MM:SS format
    match=re.search(pattern, string)
    return match.group() if match else None

def transtime(timing):
    temptime=datetime.strptime(timing,"%H:%M:%S").timetuple()
    return temptime.tm_hour*3600+temptime.tm_min*60+temptime.tm_sec

def timebeautifer(timing):
    return time.strftime("%Y-%m-%dT%H:%M:%SZ", timing)

def getrole(string):
    return re.search(r'\*\*(.*?)\*\*',string).group(1)

if not (os.path.isdir("data\\player")):
    os.mkdir("data\\player")     #making corresponding dir

colname=["id","game_length","game_starttime(UTC)","game_endtime(UTC)","player_count","players"      #All times are UTC+0, game_length is measured in second
        ,"player_alive","player_died","player_win","player_lose","first_victim"]
playercolname=["gameid","wingame","role","alive","diedfb"]

filename="rawdata.ob"
with open (f"data\\{filename}", "rb") as fp:
    readlist=pickle.load(fp)

try:
    gamedatabase=pd.read_csv("data\\gamedata.csv")
except:
    gamedatabase=pd.DataFrame(columns=colname)

gameplayed=len(readlist)    #Games played during the given period
for items in readlist:
    players=[]
    playeralive=[]
    playerdied=[]
    playerwon=[]
    playerlost=[]
    waitingforwriteplayer=[]

    endtime=items["time"]
    endtime=calendar.timegm(endtime.timetuple())
    id=((int)(endtime))
    if id in gamedatabase["id"].values:     #ID=game ending time in UTC+0, unique and will skip if there is already record on that
        continue

    splitline=items["content"].split("\n")

    iterlist=1  #starts with 1 since first line is finished
    while (iterlist<len(splitline)-1):
        noname=re.sub(r'\[.*?\]', '',splitline[iterlist]).strip()     #Delete the name of the user, we only need user id (maybe add back them later)
        try:
            playerid=get_all_integers(noname)[0]
        except:
            playerid=99999999999    #Assumes it is a ghost player if can't get there id (i.e they left the group during the game)
        players.append(playerid)
        if "alive" in noname:
            playeralive.append(playerid)
            playerisalive=True
        elif "dead" in noname:      #should be "else" in this line, but just prevent sth bad happened first
            playerdied.append(playerid)
            playerisalive=False
        else:
            print("Something went wrong. It should not reach this line")

        if "won" in noname:
            playerwon.append(playerid)
            playeriswon=True
        elif "lost" in noname:
            playerlost.append(playerid)
            playeriswon=False
        else:
            print("Something went wrong. It should not reach this line")

        rolename=getrole(noname)[:-1]
        try:        #get first blood victim. (Only one, maybe need a better way to get, which includes scrapping more bot messages) (rip my idea)
            fb=playerdied[0]
        except:
            fb=""
        if (fb==playerid):
            playerisfb=True
        else:
            playerisfb=False

        waitingforwriteplayer.append([playerid,[id,playeriswon,rolename,playerisalive,playerisfb]])
        iterlist+=1

    playercount=len(players)    #playercount

    timeplayed=get_time(splitline[-1])      #Should be str for this, need to change that
    transtimeplayed=transtime(timeplayed)
    tempstarttime=(endtime-transtimeplayed)
    starttime=time.gmtime(tempstarttime)

    newrow=pd.DataFrame([[id,transtimeplayed,timebeautifer(starttime),timebeautifer(items["time"].timetuple()),playercount,players,playeralive,playerdied,playerwon,playerlost,fb]],columns=colname)
    gamedatabase=pd.concat([gamedatabase,newrow],ignore_index=True)

    for items in waitingforwriteplayer:     #input into user database for each game
        try:
            user=pd.read_csv(f"data\\player\\{items[0]}.csv")
        except:
            user=pd.DataFrame(columns=playercolname)
        if (items[1][0] in user["gameid"].values):
            continue
        else:
            newgameforuser=pd.DataFrame([items[1]],columns=playercolname)
            user=pd.concat([user,newgameforuser],ignore_index=True)
            user.to_csv(f"data\\player\\{items[0]}.csv",index=False)

gamedatabase.to_csv("data\\gamedata.csv",index=False)
print(gamedatabase.to_markdown())

In [14]:
#Create user (all) database
allusercol=["userid","gameplayed","winrate","common_role","survivalrate","fbrate","fbcount"]
onlyfiles=[f for f in listdir("data\\player") if isfile(join("data\\player", f))]

user_all=pd.DataFrame(columns=allusercol)   #Don't need to read previous file as you have to update it anyway if you have new games.

for items in onlyfiles:
    userid=items[:-4]
    record=pd.read_csv(f"data\\player\\{items}")
    totalgames=record.shape[0]
    winrate=record[record.wingame==True].shape[0]/totalgames
    commonrole=record["role"].mode()[0]
    survivalrate=record[record.alive].shape[0]/totalgames
    fbrate=record[record.diedfb==True].shape[0]/totalgames
    fbcount=record[record.diedfb==True].shape[0]

    insert=pd.DataFrame([[userid,totalgames,winrate,commonrole,survivalrate,fbrate,fbcount]],columns=allusercol)
    user_all=pd.concat([user_all,insert],ignore_index=True)

print(user_all.to_markdown())
user_all.to_csv("data\\userall.csv",index=False)

|     |      userid |   gameplayed |   winrate | common_role        |   survivalrate |    fbrate |   fbcount |
|----:|------------:|-------------:|----------:|:-------------------|---------------:|----------:|----------:|
|   0 |  1007570205 |           14 |  0.714286 | a cultist          |       0.714286 | 0         |         0 |
|   1 |  1018375312 |            2 |  1        | a mason            |       0.5      | 0         |         0 |
|   2 |  1080215864 |            2 |  1        | the fool           |       1        | 0         |         0 |
|   3 |  1084817391 |           20 |  0.65     | a mason            |       0.75     | 0.05      |         1 |
|   4 |   116330040 |            1 |  0        | a werewolf         |       0        | 0         |         0 |
|   5 |  1182603889 |           56 |  0.660714 | a villager         |       0.535714 | 0.107143  |         6 |
|   6 |  1285660197 |            1 |  0        | a werewolf         |       0        | 1         |         1 |
|

In [15]:
temp=user_all[user_all.gameplayed>=50].sort_values(by="fbrate",ascending=False)
print(temp.to_markdown())

#First target done

|    |     userid |   gameplayed |   winrate | common_role   |   survivalrate |    fbrate |   fbcount |
|---:|-----------:|-------------:|----------:|:--------------|---------------:|----------:|----------:|
| 28 |  226769457 |          102 |  0.647059 | the augur     |       0.460784 | 0.215686  |        22 |
|  8 | 1339911576 |          348 |  0.635057 | a villager    |       0.399425 | 0.212644  |        74 |
| 37 |  273398478 |           93 |  0.655914 | a mason       |       0.44086  | 0.182796  |        17 |
| 39 |  290102454 |          156 |  0.615385 | a mason       |       0.519231 | 0.166667  |        26 |
| 31 |  234372873 |          310 |  0.625806 | a villager    |       0.467742 | 0.158065  |        49 |
| 45 |  377459053 |          165 |  0.593939 | a mason       |       0.478788 | 0.157576  |        26 |
| 29 |  229729531 |          344 |  0.622093 | a villager    |       0.44186  | 0.142442  |        49 |
| 13 |  167932826 |          554 |  0.611913 | a villager    |  

In [26]:
#Build up the role database first (You need to make it by hands)
#note that all comparison are lower cases, so you need to compare like this: useractualaction=roledatabase[roledatabase["name"].str.contains(userfinalrole,case=False,na=False)]["act"].values[0]
#maybe you can change all letters to be small letter, but this one is built according to the official names
rolecolname=["name","abbre_name","act","team"]

try:
    roledatabase=pd.read_csv("data\\roles.csv")
except:
    roledatabase=pd.DataFrame(columns=rolecolname)

    #long,long statement to actually add all the roles
    #make ota to be specific act claim instead of "night", which is different from the bot
    #for making full-thief working, can just ignoring games with thief
    roledatabase=pd.concat([roledatabase,(pd.DataFrame([["a Villager",["vg","farmer","lowly"],"no","vg"]],columns=rolecolname))],ignore_index=True)
    roledatabase=pd.concat([roledatabase,(pd.DataFrame([["a Werewolf",["ww","wolf"],"night","ww"]],columns=rolecolname))],ignore_index=True)
    roledatabase=pd.concat([roledatabase,(pd.DataFrame([["the Drunk",["drunk","alcohol"],"no","vg"]],columns=rolecolname))],ignore_index=True)
    roledatabase=pd.concat([roledatabase,(pd.DataFrame([["the Seer",["seer"],"night","vg"]],columns=rolecolname))],ignore_index=True)
    roledatabase=pd.concat([roledatabase,(pd.DataFrame([["the Cursed",["cursed"],"no","vg"]],columns=rolecolname))],ignore_index=True)
    roledatabase=pd.concat([roledatabase,(pd.DataFrame([["the Harlot",["harlot","hoe"],"night","vg"]],columns=rolecolname))],ignore_index=True)
    roledatabase=pd.concat([roledatabase,(pd.DataFrame([["the Beholder",["beholder","bh"],"no","vg"]],columns=rolecolname))],ignore_index=True)
    roledatabase=pd.concat([roledatabase,(pd.DataFrame([["the Gunner",["gunner"],"day","vg"]],columns=rolecolname))],ignore_index=True)
    roledatabase=pd.concat([roledatabase,(pd.DataFrame([["the Traitor",["traitor"],"no","vg"]],columns=rolecolname))],ignore_index=True)
    roledatabase=pd.concat([roledatabase,(pd.DataFrame([["the Guardian Angel",["guardian angel","ga","angel"],"night","vg"]],columns=rolecolname))],ignore_index=True)

    roledatabase=pd.concat([roledatabase,(pd.DataFrame([["the Detective",["detective","dete","det"],"day","vg"]],columns=rolecolname))],ignore_index=True)
    roledatabase=pd.concat([roledatabase,(pd.DataFrame([["the Apprentice Seer",["apprentice seer","app seer","app see","app"],"no","vg"]],columns=rolecolname))],ignore_index=True)
    roledatabase=pd.concat([roledatabase,(pd.DataFrame([["a Cultist",["cultist","cult"],"night","cult"]],columns=rolecolname))],ignore_index=True)
    roledatabase=pd.concat([roledatabase,(pd.DataFrame([["the Cultist Hunter",["cultist hunter","ch"],"night","vg"]],columns=rolecolname))],ignore_index=True)
    roledatabase=pd.concat([roledatabase,(pd.DataFrame([["the Wild Child",["wild child","wc"],"ota","vg"]],columns=rolecolname))],ignore_index=True)
    roledatabase=pd.concat([roledatabase,(pd.DataFrame([["the Fool",["fool","uno"],"night","vg"]],columns=rolecolname))],ignore_index=True)
    roledatabase=pd.concat([roledatabase,(pd.DataFrame([["a Mason",["mason"],"no","vg"]],columns=rolecolname))],ignore_index=True)
    roledatabase=pd.concat([roledatabase,(pd.DataFrame([["the Doppelganger",["doppelganger","dg"],"ota","other"]],columns=rolecolname))],ignore_index=True)   #the only thing I changed as we don't want "ä"
    roledatabase=pd.concat([roledatabase,(pd.DataFrame([["Cupid",["cupid"],"ota","vg"]],columns=rolecolname))],ignore_index=True)
    roledatabase=pd.concat([roledatabase,(pd.DataFrame([["the Hunter",["hunter"],"no","vg"]],columns=rolecolname))],ignore_index=True)

    roledatabase=pd.concat([roledatabase,(pd.DataFrame([["the Serial Killer",["serial killer","sk"],"night","other"]],columns=rolecolname))],ignore_index=True)
    roledatabase=pd.concat([roledatabase,(pd.DataFrame([["the Tanner",["tanner","red"],"no","other"]],columns=rolecolname))],ignore_index=True)
    roledatabase=pd.concat([roledatabase,(pd.DataFrame([["the Mayor",["mayor","mayo"],"day","vg"]],columns=rolecolname))],ignore_index=True)
    roledatabase=pd.concat([roledatabase,(pd.DataFrame([["the Prince",["prince","princess","crown"],"no","vg"]],columns=rolecolname))],ignore_index=True)
    roledatabase=pd.concat([roledatabase,(pd.DataFrame([["the Sorcerer",["sorcerer","sorc"],"night","ww"]],columns=rolecolname))],ignore_index=True)
    roledatabase=pd.concat([roledatabase,(pd.DataFrame([["the Clumsy Guy",["clumsy guy","clumsy","cg"],"no","vg"]],columns=rolecolname))],ignore_index=True)
    roledatabase=pd.concat([roledatabase,(pd.DataFrame([["the Blacksmith",["blacksmity","bs"],"day","vg"]],columns=rolecolname))],ignore_index=True)
    roledatabase=pd.concat([roledatabase,(pd.DataFrame([["the Alpha Wolf",["alpha wolf","alpha"],"night","ww"]],columns=rolecolname))],ignore_index=True)
    roledatabase=pd.concat([roledatabase,(pd.DataFrame([["the Wolf Cub",["wolf cub","cub"],"night","ww"]],columns=rolecolname))],ignore_index=True)

    roledatabase=pd.concat([roledatabase,(pd.DataFrame([["the Sandman",["sandman","sing"],"day","vg"]],columns=rolecolname))],ignore_index=True)
    roledatabase=pd.concat([roledatabase,(pd.DataFrame([["the Oracle",["oracle","oralb","oral b","ora"],"night","vg"]],columns=rolecolname))],ignore_index=True)
    roledatabase=pd.concat([roledatabase,(pd.DataFrame([["the WolfMan",["wolfman","wm"],"no","vg"]],columns=rolecolname))],ignore_index=True)
    roledatabase=pd.concat([roledatabase,(pd.DataFrame([["the Lycan",["lycan"],"night","ww"]],columns=rolecolname))],ignore_index=True)
    roledatabase=pd.concat([roledatabase,(pd.DataFrame([["the Pacifist",["pacifist","paci","burden"],"day","vg"]],columns=rolecolname))],ignore_index=True)
    roledatabase=pd.concat([roledatabase,(pd.DataFrame([["the Wise Elder",["wise elder","we","wise"],"no","vg"]],columns=rolecolname))],ignore_index=True)
    roledatabase=pd.concat([roledatabase,(pd.DataFrame([["the Thief",["thief"],"ota","other"]],columns=rolecolname))],ignore_index=True)
    roledatabase=pd.concat([roledatabase,(pd.DataFrame([["the Troublemaker",["troublemaker","tm"],"day","vg"]],columns=rolecolname))],ignore_index=True)
    roledatabase=pd.concat([roledatabase,(pd.DataFrame([["the Chemist",["chemist","chem"],"night","vg"]],columns=rolecolname))],ignore_index=True)
    roledatabase=pd.concat([roledatabase,(pd.DataFrame([["the Snow Wolf",["snow wolf","snow"],"night","ww"]],columns=rolecolname))],ignore_index=True)

    roledatabase=pd.concat([roledatabase,(pd.DataFrame([["the Grave Digger",["grave digger","gd"],"no","vg"]],columns=rolecolname))],ignore_index=True)
    roledatabase=pd.concat([roledatabase,(pd.DataFrame([["the Arsonist",["arsonist","arso"],"night","other"]],columns=rolecolname))],ignore_index=True)
    roledatabase=pd.concat([roledatabase,(pd.DataFrame([["the Augur",["augur","bird"],"no","vg"]],columns=rolecolname))],ignore_index=True)    

print(roledatabase.to_markdown())
roledatabase.to_csv("data\\roles.csv",index=False)

|    | name                | abbre_name                                        | act   | team   |
|---:|:--------------------|:--------------------------------------------------|:------|:-------|
|  0 | a Villager          | ['vg', 'farmer', 'lowly']                         | no    | vg     |
|  1 | a Werewolf          | ['ww', 'wolf']                                    | night | ww     |
|  2 | the Drunk           | ['drunk', 'alcohol']                              | no    | vg     |
|  3 | the Seer            | ['seer']                                          | night | vg     |
|  4 | the Cursed          | ['cursed']                                        | no    | vg     |
|  5 | the Harlot          | ['harlot', 'hoe']                                 | night | vg     |
|  6 | the Beholder        | ['beholder', 'bh']                                | no    | vg     |
|  7 | the Gunner          | ['gunner']                                        | day   | vg     |
|  8 | the Traitor  

Step 2:

In [ ]:
#Second target: extract first claims for players, compare it with the end-game role
group="@WerewolfAmnesia"
searchmessage=""

if not (os.path.isdir("data\\games")):
    os.mkdir("data\\games")     #making corresponding dir

gamedatabase=pd.read_csv("data\\gamedata.csv")

for index,items in gamedatabase.iterrows():
    datemin=items["game_starttime(UTC)"].replace('Z', '+00:00')
    datemax=items["game_endtime(UTC)"].replace('Z', '+00:00')

    print(f"{items['id']}: {datemin}-{datemax}")
    if not (os.path.isfile(f"data\\games\\{items['id']}.ob")):
        await scrapwithnofromuser(api_id,api_hash,group,datemin,datemax,searchmessage,"games\\"+str(items['id'])+".ob")
        print(f"{items['id']} done")

In [11]:
#Add the "gameclaim" and "claimlie" columns to user(single) database
onlyfiles=[f for f in listdir("data\\player") if isfile(join("data\\player", f))]

for items in onlyfiles:
    target=pd.read_csv(f"data\\player\\{items}")
    if "gameclaim" not in target.columns:
        target["gameclaim"]=None
    if "claimlie" not in target.columns:
        target["claimlie"]=None
    if "claimtype" not in target.columns:
        target["claimtype"]=None
    
    target.to_csv(f"data\\player\\{items}",index=False)

In [42]:
#Add the "nickname" columns to user(all) database
user_all=pd.read_csv("data\\userall.csv")
user_all["nickname"]=""

#rip now need to input user nickname one by one
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==1007570205,pd.Series([["loki"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==1018375312,pd.Series([["senku"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==1080215864,pd.Series([["septyan"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==1084817391,pd.Series([["actualafk","afk"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==116330040,pd.Series([["shan"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==1182603889,pd.Series([["nityam"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==1285660197,pd.Series([["dini"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==1334471689,pd.Series([["neel"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==1339911576,pd.Series([["emo"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==140409452,pd.Series([["kellyysw","kelly"]]*user_all.shape[0]))

user_all["nickname"]=user_all["nickname"].mask(user_all.userid==1467965424,pd.Series([["initial44"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==1624246806,pd.Series([["rain"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==162473251,pd.Series([["van"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==167932826,pd.Series([["kirs","sis"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==1690934170,pd.Series([["mt","m.t"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==1693420924,pd.Series([["shun","amalka","froggen"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==1813524300,pd.Series([[]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==1844706311,pd.Series([["bipedal"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==187680781,pd.Series([["erl"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==1897844890,pd.Series([["reem"]]*user_all.shape[0]))

user_all["nickname"]=user_all["nickname"].mask(user_all.userid==1943358703,pd.Series([["piyux"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==194893538,pd.Series([["pree"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==1993058305,pd.Series([["kambing"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==212688780,pd.Series([["zy"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==2129052793,pd.Series([[]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==215079199,pd.Series([["zann"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==224565582,pd.Series([["aasim","bakaasim","bakasim"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==224704729,pd.Series([["miracle"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==226769457,pd.Series([["touya","hanif"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==229729531,pd.Series([["miku"]]*user_all.shape[0]))

user_all["nickname"]=user_all["nickname"].mask(user_all.userid==232776540,pd.Series([["adz"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==234372873,pd.Series([["carol","crolli","natasha"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==234938824,pd.Series([["fuuka"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==235886496,pd.Series([["faa"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==255752617,pd.Series([["ran"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==259012649,pd.Series([["jazmine"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==262025635,pd.Series([["natsuko","natsu","sakura"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==273398478,pd.Series([["integer","katsu"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==273492655,pd.Series([["isaac","racoon"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==290102454,pd.Series([["sange"]]*user_all.shape[0]))

user_all["nickname"]=user_all["nickname"].mask(user_all.userid==295152997,pd.Series([["ludwig","ape"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==300889398,pd.Series([["amat"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==33517996,pd.Series([["jj"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==350553835,pd.Series([["tj"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==373668057,pd.Series([["kim"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==377459053,pd.Series([["puchi"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==382292415,pd.Series([["izzi","ez"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==39747790,pd.Series([["risa"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==398181049,pd.Series([["dh"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==400589207,pd.Series([["ytk"]]*user_all.shape[0]))

user_all["nickname"]=user_all["nickname"].mask(user_all.userid==436968848,pd.Series([["s33dz"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==442893515,pd.Series([["sorenz","soren"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==447896273,pd.Series([["ikki"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==455933988,pd.Series([["adri"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==486416416,pd.Series([["nev","jenna"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==5004310648,pd.Series([["selene"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==519782620,pd.Series([["vivian"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==52710230,pd.Series([["benjen","stark"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==540755488,pd.Series([["pingu"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==541766689,pd.Series([["vozter","voz"]]*user_all.shape[0]))

user_all["nickname"]=user_all["nickname"].mask(user_all.userid==5417793882,pd.Series([["tidal"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==5653374410,pd.Series([["zera"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==5667299838,pd.Series([["fayrouz"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==5764456589,pd.Series([["kyeo"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==578322912,pd.Series([["sophie"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==5795859799,pd.Series([["eren"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==5848950899,pd.Series([["yosra"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==5978581693,pd.Series([["sancus"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==6208061037,pd.Series([["ujimayong"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==626307008,pd.Series([["cc","ribbon"]]*user_all.shape[0]))

user_all["nickname"]=user_all["nickname"].mask(user_all.userid==630150614,pd.Series([["shing"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==6538455109,pd.Series([["asahi"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==6624187611,pd.Series([["don","pabs"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==666685999,pd.Series([["rob"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==6780433160,pd.Series([["kopenhag"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==7055130331,pd.Series([["amy"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==706049195,pd.Series([["steffi"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==7167043742,pd.Series([["issicle"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==7266875150,pd.Series([["mafaaw"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==7284589225,pd.Series([["barath"]]*user_all.shape[0]))

user_all["nickname"]=user_all["nickname"].mask(user_all.userid==74312501,pd.Series([["kai"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==7521154546,pd.Series([["luvnuraa"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==7566043263,pd.Series([["al","siddiq"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==7571056171,pd.Series([["peony"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==7604049074,pd.Series([["lacto"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==7730569797,pd.Series([["2s"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==7826380931,pd.Series([["jaslyn"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==793484112,pd.Series([["ghost"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==7971980955,pd.Series([[]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==7991056806,pd.Series([["al","siddiq"]]*user_all.shape[0]))      #note this is the same account as 7566043263, but this one is deleted account

user_all["nickname"]=user_all["nickname"].mask(user_all.userid==8136634372,pd.Series([["aya","ayaana"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==8185993052,pd.Series([["aria"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==871166732,pd.Series([["mhtb"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==879099173,pd.Series([["hashim"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==88890925,pd.Series([["c","notnatural","clover"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==888931745,pd.Series([["yiee"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==89887899,pd.Series([["ben","benjamin"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==905232905,pd.Series([["ruh"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==942520992,pd.Series([["sj"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==973655454,pd.Series([["ju"]]*user_all.shape[0]))
user_all["nickname"]=user_all["nickname"].mask(user_all.userid==99999999999,pd.Series([[]]*user_all.shape[0]))

print(user_all.head().to_markdown())
user_all.to_csv("data\\userall.csv",index=False)

|    |     userid |   gameplayed |   winrate | common_role   |   survivalrate |   fbrate |   fbcount | nickname             |
|---:|-----------:|-------------:|----------:|:--------------|---------------:|---------:|----------:|:---------------------|
|  0 | 1007570205 |           14 |  0.714286 | a cultist     |       0.714286 |     0    |         0 | ['loki']             |
|  1 | 1018375312 |            2 |  1        | a mason       |       0.5      |     0    |         0 | ['senku']            |
|  2 | 1080215864 |            2 |  1        | the fool      |       1        |     0    |         0 | ['septyan']          |
|  3 | 1084817391 |           20 |  0.65     | a mason       |       0.75     |     0.05 |         1 | ['actualafk', 'afk'] |
|  4 |  116330040 |            1 |  0        | a werewolf    |       0        |     0    |         0 | ['shan']             |


In [ ]:
async def test(apiid,apihash):
    async with TelegramClient('collectchat', apiid, apihash) as client:
        await client.get_participants("@WerewolfAmnesia")
        await client.get_dialogs()

await test(api_id,api_hash)

In [5]:
#should we do the ota part? (since their roles have a high chance to change and can be considered as lie easily)
nlp=spacy.load("en_core_web_sm")
targetedotawords="stolen infected ota bitten dg dged wc"
target=nlp(targetedotawords)
targettokens=[token.lemma_ for token in target]

def changepronoun(word):    #only take 1 word for input
    word_map={
        "me": "i",
        "him": "he",
        "her": "she",
        "us": "we",
        "them": "they",
        "mine": "i",
        "hers": "she",
        "his": "he",
        "ours": "we",
        "theirs": "they",
        "myself": "i",
        "himself": "he",
        "herself": "she",
        "ourselves": "we",
        "themselves": "they"
    }
    if (word) in word_map:
        return word_map[word]
    else:
        return word
    
def changeword(word):    #only take 1 word for input, change for typo
    word_map={
        "nil":"no",
        "egg":"act",
        "mo":"no",
        "ack":"act",
        "role":"act",
        "action":"act",
        "move":"act",
        "button":"act",
        "not":"no",
        "ni":"no"
    }
    if (word) in word_map:
        return word_map[word]
    else:
        return word
    
async def findafterchat(gameid,people,apiid,apihash,group):   #default 3 minutes
    outputlist=[]
    allgamedata=pd.read_csv("data\\gamedata.csv")
    timestart=allgamedata[allgamedata.id==gameid]["game_endtime(UTC)"].values[0]    #using endtime here
    #do the date change format here
    timestart=datetime.fromisoformat(timestart.replace('Z', '+00:00')).replace(tzinfo=timezone.utc)    #UTC time
    timeend=timestart+timedelta(minutes=3)

    async with TelegramClient('collectchat', apiid, apihash) as client:
        ''' This block only need to run once, if you haven't run it then try this
        #await client.get_participants("@WerewolfAmnesia")
        #await client.get_dialogs()
        '''
        try:
            targetuser=await client.get_entity(people)
        except:
            return "N/A"    #can't find the user

        async for message in client.iter_messages(group,offset_date=timeend,from_user=targetuser):
            if timestart <= message.date <= timeend:
                cleantext=clean(message.text, no_emoji=True)       #False, as want to keep the emojis for investigation (emoji also useful for role claim)
                outputlist.append({
                    "time":message.date,
                    "content":cleantext,
                    "sender_id":message.from_id
                })
            elif timestart>message.date:
                break    
    
    for things in outputlist:
        doc=nlp(things["content"])
        lemmatized_tokens=[token.lemma_ for token in doc]
        for words in lemmatized_tokens:
            if words in targettokens:
                return things["content"]    #means we find the ota evidence
            
    return "N/A"    #means we didn't find the ota evidence


class createdyvars:
    def __init__(self):
        pass    #set the variables from outside
    
    def getrole(self,user,game):   #return a role (str)
        df=getattr(self,user)
        return df[df.gameid==game]["role"].values[0]

    def changeclaimlie(self,user,game,claim,lie,typing):  #input claims and lies to the database for a user for a game
        df=getattr(self,user)
        df.loc[df['gameid']==game,'gameclaim']=str(claim)  # Update gameclaim
        df.loc[df['gameid']==game,'claimlie']=lie          # Update claimlie
        df.loc[df['gameid']==game,'claimtype']=typing         # Update claimtype
        setattr(self,user,df)

    def readdf(self,user):  #this one for reading data in the stored variable, test only
        df=getattr(self,user)
        print(df.to_markdown())
    
    def exportdf(self,user):    #export a dataframe inside the class (since it stores players only, hence filepath include here)
        df=getattr(self,user)
        df.to_csv(f"data\\player\\{user}.csv",index=False)

In [6]:
allusers=createdyvars()  #This one stores all the csv from "data\player"
onlyplayers=[f for f in listdir("data\\player") if isfile(join("data\\player", f))]
for items in onlyplayers:
    setattr(allusers, f'{items[:-4]}',pd.read_csv(f"data\\player\\{items}"))

gamedatabase=pd.read_csv("data\\gamedata.csv",converters={"players":literal_eval})
roledatabase=pd.read_csv("data\\roles.csv",converters={"abbre_name":literal_eval})
user_all=pd.read_csv("data\\userall.csv",converters={"nickname":literal_eval})
abbrolename=roledatabase["abbre_name"]
nicknamelist=[]
loadedmodel=0

#loading our model here
try:
    model_dir="./bert_binary_v2final"   #v2 final, accuracy improved
    tokenizer=BertTokenizer.from_pretrained(model_dir)
    model=BertForSequenceClassification.from_pretrained(model_dir)
    # Move model to appropriate device (e.g., GPU if available)
    device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()  # Set to evaluation mode
    loadedmodel=1
except:
    print("*"*30)
    print("No trained model found. Will process without that.")
    print("Please train the model after getting the result, then run this block again.")
    print("*"*30)

#extract all the things in nickname to check if exists in sentences in the game conversation for loop
for names in user_all["nickname"]:
    nicknamelist+=names

onlygames=[f for f in listdir("data\\games") if isfile(join("data\\games", f))]
#for loop starts here (looping all the game conversations)
for games in onlygames:
    readlist=[]
    gameid=games[:-3]
    gameid=int(gameid)
    with open (f"data\\games\\{games}", "rb") as fp:
        readlist=pickle.load(fp)

    for items in gamedatabase["players"][gamedatabase.id==gameid].values[0]:
        claims=""       #storage for one player's claim
        claimedaction=""    #use this to determine if player is lying, will only store ["no","day","night"], or nothing if no claims
        newlist=[x for x in readlist if x["sender_id"].user_id == items]
        for moreitems in reversed(newlist): #reversed as wanted to find the earliest msg first
            tempclaim=""    #need a tempclaim here to prevent things being changed in claims which is not supposed
            output=nlp(moreitems["content"].lower().replace("\n", " "))     #change to lower cases, also remove all "\n"
            lastword=""  #maybe useful, try store the last word

            #Try break the sentence here?
            for words in output:                #Looping one sentence
                if (words.pos_=="SCONJ"):       #Skip sentences that contains "if", those conditional clause
                    tempclaim=""                   #Reset claims as that is not the claim of the player
                    #lastword=wordchanged
                    break
                if (words.pos_=="PRON"):
                    if (changepronoun(words.text)!="i"):    #Skip sentences that having pronouns not being myself, probably referring to others
                        tempclaim=""
                        #lastword=wordchanged
                        break
                if str(words) in nicknamelist:   #skipping sentences that have words in the nickname list, mostly due to referring to other users
                    tempclaim=""
                    break
                
                wordchanged=changeword(words.text)    #change the word (typo will change here)
                wordchanged=changepronoun(wordchanged)    #I think we should also change pronoun here (handling cases for "not me")
                if (lastword=="no" and wordchanged=="act"):
                    claimedaction="no"
                    tempclaim=output
                elif (lastword=="no" and wordchanged=="i"):
                    if tempclaim=="":   #don't handle this case first as there are more meanings for (not me), and now we are focusing on role claims
                        lastword=wordchanged
                        break
                    else:   #Skip sentences that having pronouns being myself and after "no"
                        tempclaim=""
                        lastword=wordchanged
                        break                
                else:
                    for names in abbrolename:
                        if (wordchanged=="ota"):    #ota does not belongs to any abbre_names for roles so put it here
                            if lastword!="no":
                                claimedaction="ota"
                                tempclaim=output
                        if (wordchanged in names):
                            if lastword!="no":
                                claimedaction=roledatabase["act"][roledatabase["abbre_name"].apply(lambda x: wordchanged in x)].values[0]   #maybe use lambda function here for checking thief/other otas
                                tempclaim=output

                lastword=wordchanged
            
            if(tempclaim!=""):
                #check if the sentence is point to themselves using our model trained
                if loadedmodel:
                    inputsentence=" ".join(map(changeword, tempclaim.text.split()))
                    inputs=tokenizer(
                        inputsentence,
                        padding="max_length",
                        truncation=True,
                        max_length=128,
                        return_tensors="pt"
                    )
                    # Move inputs to device
                    inputs={key: val.to(device) for key, val in inputs.items()}

                    # Predict
                    with torch.no_grad():
                        outputs=model(**inputs)
                        logits=outputs.logits
                        predictions=torch.argmax(logits, dim=-1).cpu().numpy()  # 0 or 1

                    #result
                    if predictions==1:
                        claims=tempclaim
                    else:
                        tempclaim=""
                        continue
                    
                else:
                    claims=tempclaim    #Nothing need to check with the model as model doesn't exist

                #we have claimedaction to classify
                userfinalrole=allusers.getrole(str(moreitems['sender_id'].user_id),gameid)
                #Something extra for ota need to do here
                if claimedaction=="ota":    #thief/cupid is in-game claim,wc/dg is post-game claim
                    otamsg=(await findafterchat(gameid,moreitems['sender_id'],api_id,api_hash,"@WerewolfAmnesia"))
                    if (otamsg!="N/A"):
                        allusers.changeclaimlie(str(moreitems['sender_id'].user_id),gameid,(claims.text+"\n"+otamsg),False,claimedaction)
                        break   #break after you got the conclusion (true)
                    else:
                        claims=claims.text+"\n"+"N/A"

                useractualaction=roledatabase[roledatabase["name"].str.contains(userfinalrole,case=False,na=False)]["act"].values[0]
                if (userfinalrole=="a cultist"):
                    allusers.changeclaimlie(str(moreitems['sender_id'].user_id),gameid,claims,np.nan,claimedaction)
                elif(useractualaction==claimedaction):    #not lying
                    allusers.changeclaimlie(str(moreitems['sender_id'].user_id),gameid,claims,False,claimedaction)
                else:
                    allusers.changeclaimlie(str(moreitems['sender_id'].user_id),gameid,claims,True,claimedaction)
                break   #you get the claims from user, so don't need to loop for that user again
            
        if claims=="":      #only if no claim
            allusers.changeclaimlie(str(items),gameid,"0",False,"0")    #this one is not lying (as no claims)
        

allusers.readdf("626307008")

Attempt 1 at connecting failed: TimeoutError: 
Attempt 1 at connecting failed: TimeoutError: 
Attempt 1 at connecting failed: TimeoutError: 
Attempt 1 at connecting failed: TimeoutError: 
Attempt 1 at connecting failed: TimeoutError: 
Attempt 1 at connecting failed: TimeoutError: 
Attempt 1 at connecting failed: TimeoutError: 
Attempt 1 at connecting failed: TimeoutError: 
Attempt 1 at connecting failed: TimeoutError: 
Attempt 1 at connecting failed: TimeoutError: 
Attempt 1 at connecting failed: TimeoutError: 
Attempt 1 at connecting failed: TimeoutError: 
Attempt 1 at connecting failed: TimeoutError: 
Attempt 1 at connecting failed: TimeoutError: 
Attempt 1 at connecting failed: TimeoutError: 
Attempt 2 at connecting failed: TimeoutError: 
Attempt 1 at connecting failed: TimeoutError: 
Attempt 1 at connecting failed: TimeoutError: 
Attempt 1 at connecting failed: TimeoutError: 
Attempt 2 at connecting failed: TimeoutError: 
Attempt 1 at connecting failed: TimeoutError: 
Attempt 2 at 

|     |     gameid | wingame   | role                | alive   | diedfb   | gameclaim                                            |   claimlie | claimtype   |
|----:|-----------:|:----------|:--------------------|:--------|:---------|:-----------------------------------------------------|-----------:|:------------|
|   0 | 1741356097 | True      | the grave digger    | True    | False    | gd                                                   |          0 | no          |
|   1 | 1741354005 | True      | a mason             | True    | False    | 0                                                    |          0 | 0           |
|   2 | 1741352171 | False     | the augur           | False   | False    | i'm oogway (wise elder)                              |          0 | no          |
|   3 | 1741351468 | False     | the mayor           | False   | False    | farming red cabbage and kale                         |          1 | no          |
|   4 | 1741264064 | True      | the blacksmith     

In [ ]:
#testing only
model_dir="./bert_binary_finalv2"
tokenizer=BertTokenizer.from_pretrained(model_dir)
model=BertForSequenceClassification.from_pretrained(model_dir)
# Move model to appropriate device (e.g., GPU if available)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()  # Set to evaluation mode

inputsentence=["ota","is the ota wc or dg?","ota pls spare me","really ota"]
inputs=tokenizer(
    inputsentence,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)
# Move inputs to device
inputs={key: val.to(device) for key, val in inputs.items()}

# Predict
with torch.no_grad():
    outputs=model(**inputs)
    logits=outputs.logits
    predictions=torch.argmax(logits, dim=-1).cpu().numpy()  # 0 or 1

for sent, pred in zip(inputsentence, predictions):
    print(f"Sentence: '{sent}' -> Predicted Label: {pred}")

Sentence: 'ota' -> Predicted Label: 1
Sentence: 'is the ota wc or dg?' -> Predicted Label: 0
Sentence: 'ota pls spare me' -> Predicted Label: 1
Sentence: 'really ota' -> Predicted Label: 1


In [12]:
onlyplayers=[f for f in listdir("data\\player") if isfile(join("data\\player", f))]
for items in onlyplayers:   #export all the df stored to the file
    allusers.exportdf(f"{items[:-4]}")

In [48]:
#Get some claims for training and testing
claims=[]

onlyplayers=[f for f in listdir("data\\player") if isfile(join("data\\player", f))]

if not (os.path.isfile(f"data\\texttrain.ob")):   #create list for training if not exist
    for items in onlyplayers:
        tempdf=pd.read_csv(f"data\\player\\{items}")
        claims=claims+tempdf["gameclaim"].values.tolist()
    
    claims=list(set(claims))    #1291 items
    train,test=train_test_split(claims,test_size=0.8)

    with open("data\\texttrain.ob","wb") as output:     #save the output
        pickle.dump(train, output)
    with open("data\\texttest.ob","wb") as output:
        pickle.dump(test, output)

In [ ]:
with open ("data\\texttrain.ob", "rb") as fp:
    sentences=pickle.load(fp)

label=[0,1,0,1,0,1,0,0,0,1,
       1,1,1,1,0,1,1,1,1,1,
       1,1,0,1,1,1,1,1,1,1,
       0,1,0,0,1,1,1,0,1,1,
       1,1,0,1,1,1,0,1,1,0,
       1,1,0,1,0,0,1,0,1,0,
       1,1,0,0,0,1,1,1,0,0,
       0,0,0,1,0,0,0,1,0,1,
       1,1,1,0,1,1,1,1,0,1,
       1,1,0,1,0,1,1,0,1,0,
       1,0,1,1,0,0,1,0,1,1,
       1,1,0,1,1,1,1,0,0,0,
       1,0,0,1,0,0,1,1,1,1,
       0,0,1,1,0,0,1,1,1,0,
       0,1,1,0,0,0,1,0,1,1,
       1,1,1,0,0,0,1,1,1,0,
       1,1,0,0,1,1,1,0,1,1,
       1,0,1,0,0,1,0,0,1,1,
       1,0,0,1,0,1,1,0,0,0,
       1,0,1,1,0,1,1,1,1,0,
       1,1,1,1,1,1,1,0,1,0,
       0,1,1,0,1,1,1,1,0,1,
       1,0,1,0,0,1,1,0,0,1,
       1,0,0,1,0,0,0,1,1,1,
       1,0,0,1,0,0,0,1,0,1,
       1,1,0,1,0,1,1,1]    #label it 0 or 1 for the training set,258 total,0=not talking roles about user themselves,1=talking roles about themselves

sentences+=["ota","me ota","ota is wc?","ota pls don't kill","any ota","ota here"] #adding ota to the list for ota detection
label+=[1,1,0,1,0,1]

In [ ]:
#Model training (example, using v2 model now)
#Run this only when you don't have a trained model. Skip it if you have one
testsentence=[
    "ga come clean ma house",
    "augur no ch",
    "wolf, eat miku",
    "aw no act. can't kill miku",
    "lowly",
    "is the ota wc or dg?",
    "ty cupid 😘😘",
    "nice drunk wolf",
    "i'm safe by seer",
    "oh really mason",
    "blind wolf",
    "ota"
]

if (os.path.isdir("bert_binary_final")):
    raise Exception("Model already exist")

if (os.path.isdir("bert_binary_v2final")):
    raise Exception("Model already exist")

# Sample data (small dataset with labels)
train_sentences=sentences
train_labels=label  # Binary labels: 1 = positive, 0 = negative
test_sentences=testsentence  # Test set

# Step 1: Prepare dataset
train_data={"text": train_sentences, "label": train_labels}
test_data={"text": test_sentences}
train_dataset=Dataset.from_dict(train_data)
test_dataset=Dataset.from_dict(test_data)

# Step 2: Load tokenizer and model
tokenizer=BertTokenizer.from_pretrained("bert-base-uncased")
model=BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)  # 2 classes
device=torch.device("cuda" if torch.cuda.is_available() else "cpu") #should use GPU instead of CPU
model.to(device)
print(f"Model is on: {next(model.parameters()).device}")

# Step 3: Tokenize function
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

# Tokenize datasets
tokenized_train=train_dataset.map(tokenize_function, batched=True)
tokenized_test=test_dataset.map(tokenize_function, batched=True)

# Remove unnecessary columns and set format for PyTorch
tokenized_train=tokenized_train.remove_columns(["text"]).rename_column("label", "labels")
tokenized_test=tokenized_test.remove_columns(["text"])  # No labels in test for prediction
tokenized_train.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
tokenized_test.set_format("torch", columns=["input_ids", "attention_mask"])

# Step 4: Define compute_metrics for evaluation (optional)
def compute_metrics(pred):
    labels=pred.label_ids
    preds=np.argmax(pred.predictions, axis=1)
    accuracy=accuracy_score(labels, preds)
    precision, recall, f1, _=precision_recall_fscore_support(labels, preds, average="binary")
    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}

# Step 5: Training arguments
training_args=TrainingArguments(
    output_dir="./bert_binary",
    per_device_train_batch_size=2,  # Small batch for small dataset
    num_train_epochs=5,  # Few epochs to avoid overfitting
    logging_steps=10,
    save_steps=150,
    evaluation_strategy="no",  # No eval set here; add if you split data
    learning_rate=2e-5,  # Standard for BERT fine-tuning
    weight_decay=0.01,  # Regularization
)

# Step 6: Initialize Trainer
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    compute_metrics=compute_metrics,  # Optional
)

# Step 7: Train the model
trainer.train()

# Step 8: Predict on test set
predictions=trainer.predict(tokenized_test)
pred_labels=np.argmax(predictions.predictions, axis=1)  # 0 or 1

# Output predictions
print("Test Predictions (0/1):", pred_labels.tolist())
for sent, pred in zip(test_sentences, pred_labels):
    print(f"Sentence: '{sent}' -> Predicted Label: {pred}")


output_dir="./bert_binary_final"  # Directory to save the model
# Save the model
trainer.save_model(output_dir)
# Save the tokenizer
tokenizer.save_pretrained(output_dir)
print(f"Model and tokenizer saved to {output_dir}")

In [26]:
#the result should be shown like this (this is temporary pll list)
#extra task: gather more information, inspect if no act claim but ww is lie or just no act ->ww

usocolname=["id","lierate","lieratewithnoact"]
usodf=pd.DataFrame(columns=usocolname)

onlyplayers=[f for f in listdir("data\\player") if isfile(join("data\\player", f))]
for items in onlyplayers:
    tempdf=pd.read_csv(f"data\\player\\{items}")
    rate=tempdf[tempdf.claimlie==True].shape[0]/tempdf.shape[0]
    ratenoact=tempdf[((tempdf.claimlie==True) & (tempdf.claimtype=="no"))].shape[0]/tempdf.shape[0]

    temp2=pd.DataFrame([[items[:-4],rate,ratenoact]],columns=usocolname)
    usodf=pd.concat([usodf,temp2],ignore_index=True)
usodf=usodf.sort_values(by="lierate",ascending=False)

user_all=pd.read_csv("data\\userall.csv")

if "lierate" in user_all.columns:
    user_all.drop(["lierate","lieratewithnoact"],axis=1,inplace=True)

usodf['id']=usodf['id'].astype('int64')
usodf=usodf.merge(right=user_all,how="inner",left_on="id",right_on="userid")
usodf.drop("userid",axis=1,inplace=True)
usodf=usodf.rename({"id":"userid"},axis=1)

print(usodf.to_markdown())

usodf["userid"]=usodf["userid"].astype(str)
usodf=usodf.sort_values(by="userid",ascending=True)     #sort according to userid and save it
usodf=usodf.reset_index(drop=True)
usodf.to_csv("data\\userall.csv",index=False)

|     |      userid |   lierate |   lieratewithnoact |   gameplayed |   winrate | common_role        |   survivalrate |    fbrate |   fbcount | nickname                         |
|----:|------------:|----------:|-------------------:|-------------:|----------:|:-------------------|---------------:|----------:|----------:|:---------------------------------|
|   0 |   194893538 | 1         |          0         |            1 |  1        | a villager         |       1        | 0         |         0 | ['pree']                         |
|   1 |  7730569797 | 0.5       |          0.5       |            2 |  0.5      | the drunk          |       0.5      | 0         |         0 | ['2s']                           |
|   2 |  8136634372 | 0.333333  |          0.333333  |            3 |  0.666667 | the detective      |       0.666667 | 0         |         0 | ['aya', 'ayaana']                |
|   3 |   187680781 | 0.333333  |          0.333333  |            3 |  1        | a mason            |   

In [14]:
#Some statistical testing about lying between groups
group_a=[]      #pll
group_a=group_a+usodf[((usodf["id"]==255752617) | (usodf["id"]==234372873) | (usodf["id"]==167932826)
                       | (usodf["id"]==626307008) | (usodf["id"]==235886496))]["lierate"].tolist()

templist=usodf[usodf.gameplayed>=50]["lierate"].values

remove_count={x: 0 for x in group_a}
group_b=[]      #not pll with games>=50
for item in templist:
    if item in remove_count and remove_count[item]<1:
        remove_count[item]+=1
    else:
        group_b.append(item)

print(group_a)
print(group_b)
print("-"*30)

stat, p=mannwhitneyu(group_a, group_b,alternative='greater')
print(f"U-statistic: {stat}, p-value: {p}")
print("-"*30)

statistic,p_value=kruskal(group_a,group_b)
print(f"H-statistic: {statistic:.4f}")
print(f"P-value: {p_value:.4f}")

[0.33093525179856115, 0.25483870967741934, 0.18990384615384615, 0.2148014440433213, 0.11345646437994723]
[0.25, 0.23076923076923078, 0.16312056737588654, 0.17105263157894737, 0.10303030303030303, 0.10294117647058823, 0.09497206703910614, 0.11403508771929824, 0.09210526315789473, 0.13725490196078433, 0.08484848484848485, 0.03636363636363636, 0.05084745762711865, 0.17204301075268819, 0.0603448275862069, 0.04897959183673469, 0.048, 0.03197674418604651, 0.038461538461538464, 0.03571428571428571]
------------------------------
U-statistic: 89.0, p-value: 0.00282326369282891
------------------------------
H-statistic: 7.0200
P-value: 0.0081


In [3]:
#Inspect user's stats
testdf=pd.read_csv(f"data\\player\\273398478.csv")
print(testdf.to_markdown())

|    |     gameid | wingame   | role               | alive   | diedfb   | gameclaim                         | claimlie   | claimtype   |
|---:|-----------:|:----------|:-------------------|:--------|:---------|:----------------------------------|:-----------|:------------|
|  0 | 1741007933 | True      | the beholder       | True    | False    | 0                                 | False      | 0           |
|  1 | 1740660277 | True      | the serial killer  | True    | False    | chemist tho                       | False      | night       |
|  2 | 1740659609 | True      | the seer           | False   | False    | 0                                 | False      | 0           |
|  3 | 1740574038 | True      | the wolfman        | False   | True     | 0                                 | False      | 0           |
|  4 | 1740498587 | True      | the wolfman        | False   | False    | must be ota, yes.                 | True       | ota         |
|    |            |           |          

In [16]:
#Inspect game's conversation for 1 user
gameid=1740911359
user=235886496

with open (f"data\\games\\{str(gameid)}.ob", "rb") as fp:
    readlist=pickle.load(fp)

newlist=[x for x in readlist if x["sender_id"].user_id==user]
for items in newlist:
    print(items["content"])

lazy to talk
can
tf
anyone can
nice
not me
